In [2]:
import numpy as np
from numpy import ndarray
import pandas as pd
from pandas import DataFrame

scores:ndarray | None = None # 學生的分數
scores_df:DataFrame | None = None # 學生分數的DataFrame

scores = np.random.randint(50, 101, size=(50, 5))
scores_df = pd.DataFrame(scores,            
            columns=['國文', '英文', '數學', '地理', '歷史'],
            index=range(1,51))

names_df:DataFrame = pd.read_csv('students.csv')
scores_df[['姓名','性別']] = names_df[['姓名','性別']].head(n=50).values
scores_df = scores_df[['姓名', '性別','國文', '英文', '數學', '地理', '歷史']]
scores_df

,姓名,性別,國文,英文,數學,地理,歷史
1,劉彥宏,男,65,96,68,86,99
2,曾彥廷,男,86,87,59,83,88
3,吳子睿,男,99,69,68,56,71
4,蕭志明,男,75,96,63,85,77
5,王文傑,男,99,84,81,88,99
6,朱冠霖,男,85,62,96,89,94
7,謝欣蓉,女,68,79,73,86,79
8,羅淑芬,女,82,65,64,57,88
9,許美慧,女,86,91,88,96,62
10,廖志豪,男,53,85,82,77,68


In [3]:
import numpy as np
from numpy import ndarray
import pandas as pd
from pandas import DataFrame

scores:ndarray | None = None # 學生的分數
scores_df:DataFrame | None = None # 學生分數的DataFrame

scores = np.random.randint(50, 101, size=(50, 5))
scores_df = pd.DataFrame(scores,            
            columns=['國文', '英文', '數學', '地理', '歷史'],
            index=range(1,51))

names_df:DataFrame = pd.read_csv('students.csv')
scores_df[['姓名','性別']] = names_df[['姓名','性別']].head(n=50).values
scores_df = scores_df[['姓名', '性別','國文', '英文', '數學', '地理', '歷史']]
#選取欄
scores_df[['姓名','國文']]
#選取列
scores_df.iloc[5]
scores_df.iloc[5:10]

,姓名,性別,國文,英文,數學,地理,歷史
6,朱冠霖,男,60,88,66,81,72
7,謝欣蓉,女,58,65,81,59,93
8,羅淑芬,女,61,73,69,90,87
9,許美慧,女,82,86,95,72,80
10,廖志豪,男,95,64,90,84,60


In [14]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact

# -------------------------------------------------------------------------
# 1. 建立或讀取你的班級原始資料（這裡以 28 人為例）
# -------------------------------------------------------------------------
np.random.seed(42)
num_students = 28

# 模擬班級原始資料（實際使用時，可替換成你的 scores_df）
raw_data = {
    '座號': range(1, num_students + 1),
    '國文': np.random.choice([97.5, 90, 87.5, 82.5, 80, 77.5, 72.5, 70, 67.5, 62.5, 60, 52.5, 50, 47.5, 37.5, 32.5, 27.5, 25], num_students),
    '數學': np.random.randint(40, 101, num_students),
    '英語': np.random.randint(40, 101, num_students),
    '社會': np.random.randint(50, 101, num_students),
    '自然': np.random.randint(50, 101, num_students)
}
df_orig = pd.DataFrame(raw_data)
df_orig['總分'] = df_orig[['國文', '數學', '英語', '社會', '自然']].sum(axis=1)

# -------------------------------------------------------------------------
# 2. 核心邏輯：將每一科「獨立進行遞減排序」，做成底表
# -------------------------------------------------------------------------
columns_to_sort = ['國文', '數學', '英語', '社會', '自然', '總分']
sorted_columns_data = {}

for col in columns_to_sort:
    # 獨立排序並重設索引（從高分到低分）
    sorted_series = df_orig[col].sort_values(ascending=False).reset_index(drop=True)
    sorted_columns_data[col] = sorted_series

# 建立全新排名的落點表格
rank_table = pd.DataFrame(sorted_columns_data)
rank_table.index = rank_table.index + 1  # 序位從 1 開始

# -------------------------------------------------------------------------
# 3. 修正版：定義劃黑框的樣式函數（改用更安全的對照方式）
# -------------------------------------------------------------------------
def highlight_target_marks(df_data, target_scores):
    # 建立一個跟 rank_table 一模一樣大小的空白樣式表格
    styles = pd.DataFrame('', index=df_data.index, columns=df_data.columns)
    
    for col in df_data.columns:
        # 取得該目標學生的分數
        val_to_find = target_scores[col]
        # 找出在這個排序欄位中，哪些格子等於該分數
        mask = df_data[col] == val_to_find
        # 給符合的格子加上黑框
        styles.loc[mask, col] = 'border: 2.5px solid #000000; font-weight: bold; background-color: #fcfcfc;'
        
    return styles

# -------------------------------------------------------------------------
# 4. 修正版：建立互動函數
# -------------------------------------------------------------------------
def show_student_report(座號):
    # 取得該座號學生的各科實際分數
    target_scores = df_orig[df_orig['座號'] == 座號].iloc[0]
    
    print(f"=========================================")
    print(f" 🏫 福營國中成績落點表（目前選擇座號：{座號} 號）")
    print(f"=========================================")
    print(f" 國文:{target_scores['國文']} | 數學:{target_scores['數學']} | 英語:{target_scores['英語']} | 社會:{target_scores['社會']} | 自然:{target_scores['自然']} | 總分:{target_scores['總分']}")
    print(f"-----------------------------------------\n")
    
    # 修正重點：改用 Styler.applyfunc 或直接傳入整個 DataFrame 處理樣式，避免新舊版本不相容
    display(rank_table.style.apply(lambda x: highlight_target_marks(rank_table, target_scores), axis=None))
    
    print("\n💡 提示：確認無誤後，可直接在瀏覽器按 Ctrl + P 列印此區塊。")